In [1]:
pip install sec-edgar-downloader sentence-transformers pandas numpy beautifulsoup4 lxml scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.7 MB/s eta 0:00:00
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 9.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 48.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 30.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.6/484.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 41.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.5 MB/s eta 0:00:00
Using cached annotate

In [15]:
from sec_edgar_downloader import Downloader
dl = Downloader("Nandika Yadav", "nandikayadav@gmail.com")

companies = {
    "MRNA": "Moderna",
    "VRTX": "Vertex Pharmaceuticals",
    "REGN": "Regeneron",
    "LMT": "Lockheed Martin",
    "NOC": "Northrop Grumman",
    "RTX": "Raytheon Technologies",
}

for ticker in companies:
    # Pull last 5 annual reports (10-K)
    dl.get("10-K", ticker, limit=5)


In [16]:
import os

for ticker in companies:
    path = f"sec-edgar-filings/{ticker}/10-K"
    filings = os.listdir(path)
    print(ticker, "→", len(filings), "filings")


MRNA → 5 filings
VRTX → 5 filings
REGN → 5 filings
LMT → 5 filings
NOC → 5 filings
RTX → 5 filings


In [19]:
from bs4 import BeautifulSoup
import re

def find_primary_doc(filing_folder):
    """SEC filings often have multiple files; find the main 10-K document."""
    for f in os.listdir(filing_folder):
        if f.endswith(".htm") or f.endswith(".html"):
            # Usually the largest .htm file is the actual filing
            return os.path.join(filing_folder, f)
    # Fallback to full submission text file
    for f in os.listdir(filing_folder):
        if "full-submission" in f:
            return os.path.join(filing_folder, f)
    return None


In [21]:
def extract_item_1a(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    soup = BeautifulSoup(content, "lxml")
    text = soup.get_text(separator="\n")

    # Find all occurrences of "Item 1A" (case-insensitive)
    pattern = re.compile(r"item\s+1a\.?\s+risk\s+factors", re.IGNORECASE)
    matches = list(pattern.finditer(text))

    if len(matches) < 2:
        # Sometimes only one real match exists if TOC doesn't repeat the phrase exactly
        if len(matches) == 1:
            start = matches[0].end()
        else:
            return None
    else:
        # Take the SECOND match — first is usually the table of contents
        start = matches[1].end()

    # Find where Item 1B or Item 2 starts (the end boundary)
    end_pattern = re.compile(r"item\s+1b\.?\s+unresolved|item\s+2\.?\s+properties", re.IGNORECASE)
    end_match = end_pattern.search(text, start)
    end = end_match.start() if end_match else start + 50000  # fallback cap

    return text[start:end].strip()

In [23]:
results = {}

for ticker in companies:
    folder_base = f"sec-edgar-filings/{ticker}/10-K"
    results[ticker] = {}
    for filing_id in os.listdir(folder_base):
        filing_path = os.path.join(folder_base, filing_id)
        doc = find_primary_doc(filing_path)
        if doc:
            risk_text = extract_item_1a(doc)
            results[ticker][filing_id] = risk_text

# Spot check — print length and first 500 chars for each
for ticker in results:
    for filing_id, text in results[ticker].items():
        length = len(text) if text else 0
        preview = text[:300] if text else "EXTRACTION FAILED"
        print(f"\n{ticker} / {filing_id} — {length} chars")
        print(preview)


MRNA / 0001682852-26-000033 — 190313 chars
You should carefully consider the following risks and uncertainties, together with all other information in this Annual Report on Form 10-K. Any of the risk factors we describe 
below 
could adversely affect our business, financial condition or results of operations and the market price of our commo

MRNA / 0001682852-25-000022 — 186434 chars
You should carefully consider the following risks and uncertainties, together with all other information in this Annual Report on Form 10-K. Any of the risk factors we describe 
below 
could adversely affect our business, financial condition or results of operations and the market price of our commo

MRNA / 0001682852-23-000011 — 206248 chars
You should carefully consider the following risks and uncertainties, together with all other information in this Annual Report on Form 10-K. Any of the risk factors we describe below could adversely affect our business, financial condition or results of operations a

In [75]:
import re

def is_junk_title(line):
    stripped = line.strip()

    # Pure numbers (page numbers): "11", "13", "-14-", "10"
    if re.fullmatch(r"-?\d+-?", stripped):
        return True

    # "Table of Contents" and fragments — explicit phrase check
    if re.fullmatch(r"(table\s+of\s+)?contents?", stripped, re.IGNORECASE):
        return True

    # Any line that is just 1-2 short word-fragments — real risk titles
    # are always multi-word descriptive phrases like "Risks Related to..."
    words = re.findall(r"[A-Za-z]+", stripped)
    if len(words) <= 2 and len(stripped) < 20:
        return True

    # Bullet characters alone
    if stripped in ("•", "-", "*", "·"):
        return True

    # Company name as all-caps header
    if re.fullmatch(r"[A-Z\s.,]+CORPORATION", stripped):
        return True

    return False

In [97]:
def split_into_risks(risk_text):
    if not risk_text:
        return []

    # Collapse all whitespace/newlines into single spaces first —
    # don't trust line breaks as structural signals
    lines = [l.strip() for l in risk_text.split("\n") if l.strip()]
    lines = [l for l in lines if not is_junk_title(l)]

    # Rejoin everything into one continuous text, then re-split on
    # a pattern: a short Title Case phrase (the risk header) immediately
    # followed by a sentence starting with We/Our/The/etc.
    full_text = " ".join(lines)

    # Risk titles are typically short (3-15 words), often start with
    # "Risks", "Our", "We", "The" but are NOT full sentences (no period
    # before the next capital-starting sentence)
    # Pattern: end of a sentence (.) followed by a Title-Case phrase
    # of 3-15 words followed immediately by "We "/"Our "/"The "/etc.
    title_pattern = re.compile(
        r'([A-Z][a-zA-Z,\'’\- ]{10,120}?)(?=\s+(?:We|Our|The|If|Failure|Changes|Any|This|These|Such)\b[a-z])'
    )

    matches = list(title_pattern.finditer(full_text))

    risks = []
    for i, m in enumerate(matches):
        title = m.group(1).strip()
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(full_text)
        body = full_text[start:end].strip()
        if len(body) > 100:
            risks.append({"title": title, "text": body})

    return risks

In [99]:
for filing_id, text in results["LMT"].items():
    risks = split_into_risks(text)
    print(f"\n{filing_id}: {len(risks)} risks found")
    for r in risks[:3]:
        print(" -", r["title"][:80])


0001628280-26-004195: 0 risks found

0000936468-22-000008: 0 risks found

0000936468-25-000009: 0 risks found

0000936468-24-000010: 0 risks found

0000936468-23-000009: 0 risks found


In [103]:
for filing_id, text in results["MRNA"].items():
    risks = split_into_risks(text)
    print(f"\n{filing_id}: {len(risks)} risks found")
    for r in risks[:3]:
        print(" -", r["title"][:80])


0001682852-26-000033: 0 risks found

0001682852-25-000022: 0 risks found

0001682852-23-000011: 0 risks found

0001682852-22-000012: 0 risks found

0001682852-24-000015: 0 risks found


In [81]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_persistence(risks_by_year, threshold=0.75):
    """
    risks_by_year: dict mapping year -> list of risk dicts (title+text)
    Returns: list of (year, num_risks, num_persisted, num_dropped)
    """
    years = sorted(risks_by_year.keys())
    output = []

    for i in range(len(years) - 1):
        year_a, year_b = years[i], years[i+1]
        risks_a = risks_by_year[year_a]
        risks_b = risks_by_year[year_b]

        if not risks_a or not risks_b:
            continue

        texts_a = [r["title"] + " " + r["text"] for r in risks_a]
        texts_b = [r["title"] + " " + r["text"] for r in risks_b]

        emb_a = model.encode(texts_a)
        emb_b = model.encode(texts_b)

        sim_matrix = cosine_similarity(emb_a, emb_b)
        max_sims = sim_matrix.max(axis=1)  # for each risk in year_a, best match in year_b

        persisted = (max_sims >= threshold).sum()
        dropped = (max_sims < threshold).sum()

        output.append({
            "year_pair": f"{year_a}->{year_b}",
            "total_risks": len(risks_a),
            "persisted": int(persisted),
            "dropped": int(dropped),
            "drop_rate": dropped / len(risks_a)
        })

    return output

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [31]:
print(model)


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [82]:
# sec-edgar-downloader folders often look like: 0000950170-23-001234
# The "-23-" segment is the 2-digit year of FILING (not fiscal year, but close enough for today)
def extract_year_from_folder(folder_name):
    parts = folder_name.split("-")
    if len(parts) >= 2:
        yy = parts[1]
        return 2000 + int(yy)
    return None

In [85]:
all_drop_rates = {"biotech": [], "defense": []}

industry_map = {
    "MRNA": "biotech", "VRTX": "biotech", "REGN": "biotech",
    "LMT": "defense", "NOC": "defense", "RTX": "defense",
}

for ticker in companies:
    risks_by_year = {}
    folder_base = f"sec-edgar-filings/{ticker}/10-K"

    for filing_id in os.listdir(folder_base):
        year = extract_year_from_folder(filing_id)
        text = results[ticker].get(filing_id)
        risks = split_into_risks(text)
        if risks:
            risks_by_year[year] = risks

    persistence = compute_persistence(risks_by_year)

    print(f"\n=== {ticker} ({industry_map[ticker]}) ===")
    for p in persistence:
        print(p)
        all_drop_rates[industry_map[ticker]].append(p["drop_rate"])

# Summary
import numpy as np
print("\n\n=== SUMMARY ===")
print(f"Biotech avg drop rate: {np.mean(all_drop_rates['biotech']):.2%}")
print(f"Defense avg drop rate: {np.mean(all_drop_rates['defense']):.2%}")


=== MRNA (biotech) ===
{'year_pair': '2022->2023', 'total_risks': 34, 'persisted': 28, 'dropped': 6, 'drop_rate': 0.17647058823529413}
{'year_pair': '2023->2024', 'total_risks': 33, 'persisted': 30, 'dropped': 3, 'drop_rate': 0.09090909090909091}
{'year_pair': '2024->2025', 'total_risks': 45, 'persisted': 40, 'dropped': 5, 'drop_rate': 0.1111111111111111}
{'year_pair': '2025->2026', 'total_risks': 44, 'persisted': 41, 'dropped': 3, 'drop_rate': 0.06818181818181818}

=== VRTX (biotech) ===
{'year_pair': '2022->2023', 'total_risks': 36, 'persisted': 29, 'dropped': 7, 'drop_rate': 0.19444444444444445}
{'year_pair': '2023->2024', 'total_risks': 35, 'persisted': 27, 'dropped': 8, 'drop_rate': 0.22857142857142856}
{'year_pair': '2024->2025', 'total_risks': 38, 'persisted': 31, 'dropped': 7, 'drop_rate': 0.18421052631578946}
{'year_pair': '2025->2026', 'total_risks': 35, 'persisted': 19, 'dropped': 16, 'drop_rate': 0.45714285714285713}

=== REGN (biotech) ===
{'year_pair': '2022->2023', 'tot

In [91]:
def list_dropped_titles(risks_by_year, threshold=0.75):
    years = sorted(risks_by_year.keys())
    dropped_titles = []
    for i in range(len(years) - 1):
        year_a, year_b = years[i], years[i+1]
        risks_a = risks_by_year[year_a]
        risks_b = risks_by_year[year_b]
        if not risks_a or not risks_b:
            continue

        texts_a = [r["title"] + " " + r["text"] for r in risks_a]
        texts_b = [r["title"] + " " + r["text"] for r in risks_b]
        emb_a = model.encode(texts_a)
        emb_b = model.encode(texts_b)
        sim_matrix = cosine_similarity(emb_a, emb_b)
        max_sims = sim_matrix.max(axis=1)

        for idx, sim in enumerate(max_sims):
            if sim < threshold:
                dropped_titles.append((year_a, year_b, risks_a[idx]["title"], round(sim,2)))
    return dropped_titles

# Run for one company per industry
for ticker in ["MRNA", "LMT"]:
    risks_by_year = {}
    folder_base = f"sec-edgar-filings/{ticker}/10-K"
    for filing_id in os.listdir(folder_base):
        year = extract_year_from_folder(filing_id)
        text = results[ticker].get(filing_id)
        risks = split_into_risks(text)
        if risks:
            risks_by_year[year] = risks

    print(f"\n=== {ticker} dropped risks ===")
    for yA, yB, title, sim in list_dropped_titles(risks_by_year):
        print(f"  [{yA}->{yB}] sim={sim}: {title[:90]}")


=== MRNA dropped risks ===
  [2022->2023] sim=0.7099999785423279: Risks related to our pipeline, product development and regulatory review
  [2022->2023] sim=0.7200000286102295: . We have limited experience in filing and supporting the necessary applications for marke
  [2022->2023] sim=0.6200000047683716: we could face significant competition in seeking appropriate strategic collaborators and t
  [2022->2023] sim=0.6899999976158142: and elsewhere in this Annual Report on Form 10-K:
  [2022->2023] sim=0.6800000071525574: we may be incapable of producing an investigational medicine in commercial quantities at a
  [2022->2023] sim=0.7099999785423279: announcement by us or our competitors of significant acquisitions, strategic partnerships,
  [2023->2024] sim=0.699999988079071: Risks related to our pipeline, product development and regulatory review
  [2023->2024] sim=0.6499999761581421: During the 12-year period of exclusivity provided by the
  [2023->2024] sim=0.7200000286102295: shipm

In [65]:
risks_by_year = {}
folder_base = f"sec-edgar-filings/LMT/10-K"
for filing_id in os.listdir(folder_base):
    year = extract_year_from_folder(filing_id)
    text = results["LMT"].get(filing_id)
    risks = split_into_risks(text)
    if risks:
        risks_by_year[year] = risks

for r in risks_by_year[2023][:10]:
    print("-", r["title"][:100])

- Risks Related to our Reliance on Government Contracts
- f C
- f C
- f C
- f C
- Other Risks Related to our Operations
- f C
- f C
- f C
- f C


In [67]:
folder_base = f"sec-edgar-filings/LMT/10-K"
for filing_id in os.listdir(folder_base):
    year = extract_year_from_folder(filing_id)
    if year == 2023:
        text = results["LMT"].get(filing_id)
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        for i, line in enumerate(lines):
            if line == "f C" or "f C" in line:
                # print context: 2 lines before and after
                print(f"--- line {i} ---")
                for j in range(max(0,i-2), min(len(lines), i+3)):
                    marker = ">>>" if j == i else "   "
                    print(f"{marker} {lines[j]!r}")
                print()
                break  # just show the first occurrence for now

--- line 5 ---
    'We derived 73% of our total consolidated net sales from the U.S. Government in 2022, including 64% from the DoD. We expect to continue to derive most of our sales from work performed under U.S. Government contracts. Budget uncertainty, the potential for U.S. Government shutdowns, the use of continuing resolutions, and the federal debt ceiling can adversely affect our industry and the funding for our programs. If appropriations are delayed or a government shutdown were to occur and were to continue for an extended period of time, we could be at risk of program cancellations and other disruptions and nonpayment. When the U.S. Government operates under a continuing resolution, new contract and program starts are restricted and funding for our programs may be unavailable, reduced or delayed. Shifting funding priorities or federal budget compromises, also could result in reductions in overall defense spending on an absolute or inflation-adjusted basis, which could advers

KeyError: '<2023 filing_id>'